# 模型 (PyTorch)

将更详细地**了解如何创建和使用模型**。我们将使用 AutoModel 类，当你希望从 checkpoint 实例化任何模型时，使用它非常方便。

In [1]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.8 MB/s eta 0:00:00


`AutoModel` 类及其所有的相关类其实就是对库中可用的各种模型的简单包装。它是一个智能的包装，因为**它可以自动猜测你的 checkpoint 适合的模型架构**，然后**实例化一个具有相同架构的模型**。

然而，如果你知道要使用模型的类型，你**可以直接使用其架构相对应的模型类**。让我们看看如何使用 BERT 模型。

##创建 Transformer 模型

初始化 BERT 模型需要做的第一件事是加载 `Config` 对象：


In [ ]:
from transformers import BertConfig, BertModel

# Building the config
config = BertConfig()

# Building the model from the config
model = BertModel(config)

Config 对象

*  作用：**存储模型全部超参数、结构配置**，定义模型长什么样，是模型的 **“设计图纸”**。
*  网络结构：隐藏层维度`hidden_size`、注意力头数`num_attention_heads`、编码器层数`num_hidden_layers`；
*  训练 / 输入限制：最大序列长度`max_position_embeddings`、中间层维度`intermediate_size`；
各类 dropout、激活函数、词汇表大小等参数。

In [ ]:
print(config)

BertConfig {
  [...]
  "hidden_size": 768,
  "intermediate_size": 3072,
  "max_position_embeddings": 512,
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  [...]
}

##使用不同的加载方式

使用默认配置创建模型会使用随机值对其进行初始化：

In [ ]:
from transformers import BertConfig, BertModel

config = BertConfig()
model = BertModel(config)

# Model is randomly initialized!

这个模型是可以运行并得到结果的，但它会输出胡言乱语；它需要**先进行训练**才能正常使用。我们可以根据手头的任务从头开始训练模型，为了避免不必要的重复工作，能够**共享和复用已经训练过的模型**是非常重要的。

我们会将 `BertModel` 替换为等效的 `AutoModel` 类
*  替换逻辑：不用固定写BertModel这种绑定特定模型架构的类，改用通用AutoModel。
*  **摆脱 checkpoint 绑定**：BertModel只能加载 BERT 类权重文件；AutoModel会自动读取权重配套的config.json，识别底层模型架构，不用开发者手动指定模型类。
*  代码通用性：一套代码可切换不同权重（checkpoint），无需修改模型实例化代码。
*  适用前提：只要**权重是同类型任务预训练 / 微调的**（如都做文本分类），**哪怕底层模型架构不一样**（BERT、RoBERTa、DistilBERT 等），AutoModel都能自动适配加载。

现在，**此模型已经用 checkpoint 的所有权重进行了初始化。**它可以直接用于推理它训练过的任务，也**可以在新任务上进行微调**。通过使用预训练的权重进行训练，相比于从头开始训练，我们可以迅速获得比较好的结果。


In [2]:
from transformers import BertModel
#代码示例中，我们没有使用 BertConfig ，而是通过 bert-base-cased 标签加载了一个预训练模型。
model = BertModel.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


##保存模型

保存模型和加载模型一样简单—我们使用 `save_pretrained()` 方法。

In [4]:
model.save_pretrained("directory_on_my_computer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

`config.json` 文件有**构建模型架构所需的属性**。这个文件还包含一些元数据，例如 checkpoint 的来源，以及你上次保存 checkpoint 时所使用的Transformers 版本。

`pytorch_model.safetensors` 文件被称为 **state dictionary（状态字典）** ；它包含了你的模型的所有权重。

这两个文件是相辅相成的；配置文件是构建你的模型架构所必需的，而模型权重就是你的模型参数。


##使用 Transformers 模型进行推理

既然你知道了如何加载和保存模型，那么让我们尝试使用它进行一些预测。Transformer 模型只能处理数字——由 tokenizer 转化后的数字。但在我们讨论 tokenizer 之前，让我们探讨一下模型可以接受的输入是什么。

可以将输入转换为适当的框架张量，但为了帮助你了解发生了什么，我们将快速了解在将输入发送到模型之前必须做什么。



In [ ]:
sequences = ["Hello!", "Cool.", "Nice!"]

In [ ]:
encoded_sequences = [
    [101, 7592, 999, 102],
    [101, 4658, 1012, 102],
    [101, 3835, 999, 102],
]

In [ ]:
import torch

model_inputs = torch.tensor(encoded_sequences)

In [ ]:
output = model(model_inputs)